# ContractRisk — un reintento controlado tras HTTP 503

Runtime CPU, sin GPU. La corrida API 07 obtuvo 503 en su primera generación y no devolvió ninguna nota. Este notebook conserva esa evidencia y permite UNA recuperación explícita: mismo modelo, prompt, rúbrica, semilla por ancla, agenda y umbrales. Si vuelve a fallar disponibilidad, se detiene sin otro retry ni fallback ni activar pagos.

Requiere los mismos secretos GEMINI_API_KEY y WANDB_API_KEY, el mismo proyecto Free Tier sin facturación y la misma carpeta de Drive. Conserva los USD 0,3072 reservados como estimación de la petición original, no como cargo confirmado. La nueva petición debe reservar adicionalmente antes de enviarse. Ambas reservas cuentan para el mismo sublímite de calibración de USD 1 y techo total USD 5.

Verifica el hash del ZIP 07, sus fuentes y que no contenga notas válidas. Comprueba que el registro y la marca ORIGINAL siguen intactos en Drive y crea una marca NUEVA exclusiva de recuperación; no borra, reemplaza ni reinicia las anteriores. Si falta algo o cambió, detenerse y devolver resultados: no reconstruir archivos a mano.

Si la generación vuelve a estar disponible, evalúa el piloto de nueve y solo si pasa completa 38 juicios nuevos. No reutiliza como nota la petición original sin respuesta, ni selecciona entre puntuaciones. Si supera regresión, siguen pendientes auditoría humana y confirmación separada. No guarda razonamiento privado ni modifica main/Space.


In [ ]:
import os, sys, json, subprocess, tempfile
from pathlib import Path
from google.colab import userdata

ROOT = Path(tempfile.mkdtemp(prefix='contractrisk_api_503_recovery_', dir='/content')) / 'repo'
subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'Entrega/contractrisk-final',
    'https://github.com/vcastrop/Juan-Camilo-Ramon-Valentina-Castro.git', str(ROOT)], check=True)
# Required by historical-integrity unit tests; no inference from that commit is rerun.
subprocess.run(['git', '-C', str(ROOT), 'fetch', '--depth', '1', 'origin',
    '6ec8dcdf22433c632b5eb0e5539401c33b9c923d'], check=True)
subprocess.run(['git', '-C', str(ROOT), 'fetch', '--depth', '1', 'origin', '71912f1c5755790381a8e39c8a67801bcbf9243d'], check=True)
os.chdir(ROOT)
COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Commit:', COMMIT)
for name in ['GEMINI_API_KEY', 'WANDB_API_KEY']:
    try:
        value = userdata.get(name)
        if not value:
            raise ValueError('Empty secret')
        os.environ[name] = value
        del value
    except Exception:
        raise RuntimeError('Habilita el acceso de ESTE notebook al secreto ' + name + '. No pegues la clave.') from None

def run_visible(arguments, logfile):
    with open(logfile, 'w', encoding='utf-8') as log:
        p = subprocess.Popen(arguments, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            print(line, end='', flush=True)
            log.write(line); log.flush()
        status = p.wait()
    if status:
        raise RuntimeError('Proceso detenido; conservar resultados y consultar ' + logfile)


In [ ]:
run_visible([sys.executable, '-m', 'pip', 'install', '-r', 'final/requirements-api-judge.txt'], 'installation.log')
run_visible([sys.executable, '-c', "from pathlib import Path; from final.run_judge_calibration import verify_review; verify_review(Path('final/eval/calibration_review_v1')); print('Anclas aprobadas y hashes comprobados.')"], 'review_check.log')
run_visible([sys.executable, '-m', 'unittest', 'discover', '-s', 'final/tests', '-v'], 'tests_final.log')
run_visible([sys.executable, '-m', 'unittest', 'discover', '-s', 'm3/tests', '-v'], 'tests_m3.log')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BUDGET_DIR = Path('/content/drive/MyDrive/ContractRisk_final_budget')
confirmation = input('Confirma que el proyecto de GEMINI_API_KEY es Free Tier y NO tiene facturación activada. Escribe GRATUITO para continuar: ')
assert confirmation.strip() == 'GRATUITO', 'No se hará ninguna petición API sin confirmar el proyecto gratuito.'
del confirmation
print('Registro persistente de presupuesto en Drive. No borrar para repetir corridas.')


In [ ]:
RESULTS = ROOT / 'gemini_503_recovery_results'
run_visible([sys.executable, '-u', 'final/run_gemini_503_recovery.py', '--output', str(RESULTS),
    '--budget-dir', str(BUDGET_DIR), '--confirm-free-project', '--require-wandb'], 'gemini_503_recovery.log')


## Interpretación

Si aparece otra vez 503, descargar evidencia y detenerse. No borrar ninguna marca o registro de Drive. La reserva anterior permanece; no es una factura. No activar pagos, cambiar modelo ni repetir para obtener notas favorables.

Un piloto fallido con nota sí indica desacuerdo con los rangos aprobados. Un error de API o formato sin nota no permite concluir calidad de los casos pendientes. Incluso una regresión aprobada necesita auditoría humana y confirmación separada.


In [ ]:
RESULTS = ROOT / 'gemini_503_recovery_results'
for name in ['metadata.json', 'pilot_summary.json', 'regression_summary.json', 'partial_regression_summary.json']:
    path = RESULTS / name
    if path.exists():
        print(name, path.read_text(encoding='utf-8'))


In [ ]:
import shutil
from google.colab import files
RESULTS.mkdir(exist_ok=True)
(RESULTS / 'notebook_commit.txt').write_text(COMMIT + '\n', encoding='utf-8')
for logfile in ROOT.glob('*.log'):
    shutil.copy2(logfile, RESULTS / logfile.name)
for name in ['gemini_calibration_config.json', 'api_budget_policy.json']:
    shutil.copy2(ROOT / 'final' / name, RESULTS / name)
reviewed = RESULTS / 'reviewed_calibration_pack'
if not reviewed.exists():
    shutil.copytree(ROOT / 'final/eval/calibration_review_v1', reviewed)
package = shutil.make_archive(str(ROOT / 'contractrisk_final_juez_api_reintento_resultados'), 'zip', RESULTS)
files.download(package)
